# Resumable T2SMark SD3.5 one-unit GPU canary
Unexecuted experiment handoff; the project runner owns all state.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib, subprocess, sys
from google.colab import userdata
OFFICIAL_EXACT='0c1fbfd50fcd1fba135477a2c016e284d5d7914d'
assert __import__('torch').cuda.is_available()
checkout=pathlib.Path('/content/CEG-WM-Baseline-V1'); subprocess.run(['git','clone','--branch','Baseline-V1','https://github.com/RICHAAARC/CEG-WM.git',str(checkout)],check=True)
resolved_exact=subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip(); subprocess.run(['git','-C',str(checkout),'checkout','--detach',resolved_exact],check=True)
assert subprocess.check_output(['git','-C',str(checkout),'status','--porcelain'],text=True).strip()==''
official=pathlib.Path('/content/T2SMark-official'); subprocess.run(['git','clone','https://github.com/0xD009/T2SMark.git',str(official)],check=True); subprocess.run(['git','-C',str(official),'checkout','--detach',OFFICIAL_EXACT],check=True)
official_head=subprocess.check_output(['git','-C',str(official),'rev-parse','HEAD'],text=True).strip(); official_branch=subprocess.run(['git','-C',str(official),'symbolic-ref','-q','--short','HEAD'],text=True,capture_output=True).stdout.strip(); official_dirty=subprocess.check_output(['git','-C',str(official),'status','--porcelain'],text=True).strip()
print({'official_head':official_head,'detached':not bool(official_branch),'clean':not bool(official_dirty)}); assert official_head==OFFICIAL_EXACT and not official_branch and not official_dirty
subprocess.run([sys.executable,'-m','pip','install','-q','diffusers==0.32.0','transformers==4.45.2','accelerate==1.1.1','huggingface_hub==0.26.2','safetensors==0.4.5','sentencepiece==0.2.0','-e',str(checkout)],check=True)

In [ ]:
RUN_ID='t2smark_sd35_one_unit_v1'; FORCE_RERUN_ALL=False
run_dir=pathlib.Path('/content/drive/MyDrive/CEG-WM/Baseline-V1/T2SMark-Canary')/RUN_ID; child_env=dict(os.environ); child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''
command=[sys.executable,'-m','cegwm.baselines.t2smark_canary','--run-dir',str(run_dir),'--run-id',RUN_ID,'--project-exact',resolved_exact,'--official-source',str(official)]
if FORCE_RERUN_ALL: command.append('--force-rerun-all')
subprocess.run(command,cwd=checkout,env=child_env,check=True)

Only 12 real native scores constitute an engineering canary; no threshold, TPR/FPR, robustness, or paper claim.